In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

## 정규성 검증

In [2]:
"""
정규성 검증 스크립트
- Shapiro-Wilk (n <= 5000 샘플), Anderson-Darling, D'Agostino-Pearson 3가지 검정 병행
- 식별자 컬럼 제외한 모든 수치형 컬럼 대상
- 결과를 CSV + 시각적 요약으로 저장
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────────────────────────
DATA_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
OUTPUT_CSV  = os.path.join('normality_test_results.csv')
ALPHA       = 0.05          # 유의수준
SHAPIRO_N   = 5_000         # Shapiro-Wilk은 n이 크면 느리므로 샘플링
SAMPLE_SEED = 42

# 제외할 식별자 컬럼
EXCLUDE_COLS = {
    '사업자등록번호',
    '회계년도',
    '부실라벨_ICR3년',
    'M코드',
    '빅4감사',
    '회사명'
}

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
print("▶ 데이터 로드 중...")
df = pd.read_parquet(DATA_PATH)
print(f"  Shape: {df.shape}")

# 수치형 컬럼 중 식별자 제외
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]
print(f"  정규성 검정 대상 컬럼 수: {len(numeric_cols)}")

# ── 정규성 검정 함수 ──────────────────────────────────────────────────────────
def run_normality_tests(series: pd.Series, col_name: str) -> dict:
    """단일 컬럼에 대해 3가지 정규성 검정 수행"""
    # NaN 제거
    data = series.dropna().values.astype(float)
    n_total = len(series)
    n_valid = len(data)
    n_missing = n_total - n_valid
    missing_pct = round(n_missing / n_total * 100, 2)

    result = {
        'column': col_name,
        'n_total': n_total,
        'n_valid': n_valid,
        'n_missing': n_missing,
        'missing_pct': missing_pct,
        'mean': np.nan,
        'std': np.nan,
        'skewness': np.nan,
        'kurtosis': np.nan,
        # Shapiro-Wilk
        'shapiro_stat': np.nan,
        'shapiro_pvalue': np.nan,
        'shapiro_normal': np.nan,
        'shapiro_note': '',
        # Anderson-Darling
        'anderson_stat': np.nan,
        'anderson_critical_5pct': np.nan,
        'anderson_normal': np.nan,
        # D'Agostino-Pearson (K2)
        'dagostino_stat': np.nan,
        'dagostino_pvalue': np.nan,
        'dagostino_normal': np.nan,
        # 종합 판정
        'majority_normal': np.nan,
        'all_normal': np.nan,
    }

    # 유효 데이터가 너무 적으면 스킵
    if n_valid < 8:
        result['shapiro_note'] = f'유효샘플 부족 ({n_valid}개)'
        return result

    # 기술통계
    result['mean']     = round(float(np.mean(data)), 6)
    result['std']      = round(float(np.std(data, ddof=1)), 6)
    result['skewness'] = round(float(stats.skew(data)), 4)
    result['kurtosis'] = round(float(stats.kurtosis(data)), 4)  # excess kurtosis

    # ── 1. Shapiro-Wilk ──────────────────────────────────────────────────────
    # n > 5000이면 랜덤 서브샘플링 (속도 확보)
    sw_note = ''
    if n_valid > SHAPIRO_N:
        rng = np.random.default_rng(SAMPLE_SEED)
        sw_data = rng.choice(data, size=SHAPIRO_N, replace=False)
        sw_note = f'샘플링({SHAPIRO_N}/{n_valid})'
    else:
        sw_data = data
    try:
        sw_stat, sw_p = stats.shapiro(sw_data)
        result['shapiro_stat']   = round(float(sw_stat), 6)
        result['shapiro_pvalue'] = round(float(sw_p), 6)
        result['shapiro_normal'] = int(sw_p > ALPHA)
        result['shapiro_note']   = sw_note
    except Exception as e:
        result['shapiro_note'] = f'오류: {e}'

    # ── 2. Anderson-Darling ──────────────────────────────────────────────────
    try:
        ad_result = stats.anderson(data, dist='norm')
        # 5% 유의수준 인덱스 (0=15%, 1=10%, 2=5%, 3=2.5%, 4=1%)
        idx_5pct = 2
        ad_stat   = float(ad_result.statistic)
        ad_crit   = float(ad_result.critical_values[idx_5pct])
        result['anderson_stat']         = round(ad_stat, 6)
        result['anderson_critical_5pct']= round(ad_crit, 6)
        result['anderson_normal']       = int(ad_stat < ad_crit)  # 검정통계량 < 임계값이면 정규
    except Exception as e:
        result['anderson_stat'] = f'오류: {e}'

    # ── 3. D'Agostino-Pearson (K2) ───────────────────────────────────────────
    # 최소 20개 필요
    if n_valid >= 20:
        try:
            k2_stat, k2_p = stats.normaltest(data)
            result['dagostino_stat']   = round(float(k2_stat), 6)
            result['dagostino_pvalue'] = round(float(k2_p), 6)
            result['dagostino_normal'] = int(k2_p > ALPHA)
        except Exception as e:
            result['dagostino_stat'] = f'오류: {e}'

    # ── 종합 판정 ────────────────────────────────────────────────────────────
    votes = [
        result['shapiro_normal'],
        result['anderson_normal'],
        result['dagostino_normal'],
    ]
    valid_votes = [v for v in votes if isinstance(v, (int, float)) and not np.isnan(v)]
    if valid_votes:
        sum_votes = sum(valid_votes)
        result['majority_normal'] = int(sum_votes >= 2)   # 다수결 (2/3 이상)
        result['all_normal']      = int(sum_votes == len(valid_votes))  # 전체 합의

    return result


# ── 전체 컬럼 검정 실행 ───────────────────────────────────────────────────────
print("\n▶ 정규성 검정 실행 중...")
results = []
total = len(numeric_cols)

for i, col in enumerate(numeric_cols):
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"  진행: {i+1}/{total} ({(i+1)/total*100:.1f}%)")
    results.append(run_normality_tests(df[col], col))

results_df = pd.DataFrame(results)

# ── 결과 저장 ─────────────────────────────────────────────────────────────────
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ 결과 저장 완료: {OUTPUT_CSV}")

# ── 요약 출력 ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 정규성 검정 요약")
print("="*60)

n_cols = len(results_df)
n_majority_normal   = results_df['majority_normal'].sum()
n_majority_nonnorm  = n_cols - n_majority_normal
n_all_normal        = results_df['all_normal'].sum()

print(f"\n총 검정 컬럼 수   : {n_cols}")
print(f"유의수준          : α = {ALPHA}")
print(f"\n[다수결 판정 - 3개 검정 중 2개 이상 정규]")
print(f"  정규분포 O      : {int(n_majority_normal)}개  ({n_majority_normal/n_cols*100:.1f}%)")
print(f"  정규분포 X      : {int(n_majority_nonnorm)}개  ({n_majority_nonnorm/n_cols*100:.1f}%)")
print(f"\n[전체 합의 - 3개 검정 모두 정규]")
print(f"  정규분포 O      : {int(n_all_normal)}개  ({n_all_normal/n_cols*100:.1f}%)")

print("\n[각 검정별 정규 비율]")
for test_col, label in [
    ('shapiro_normal',   'Shapiro-Wilk     '),
    ('anderson_normal',  'Anderson-Darling '),
    ('dagostino_normal', "D'Agostino-Pearson"),
]:
    col_data = results_df[test_col].dropna()
    n_norm = int(col_data.sum())
    print(f"  {label}: {n_norm}/{len(col_data)} ({n_norm/len(col_data)*100:.1f}%)")

# 왜도/첨도 분포 요약
print("\n[왜도(Skewness) 절대값 분포]")
sk = results_df['skewness'].abs().dropna()
print(f"  |skew| < 0.5  (거의 대칭)   : {(sk < 0.5).sum()}개")
print(f"  0.5 ≤ |skew| < 1 (약간 비대칭): {((sk >= 0.5) & (sk < 1)).sum()}개")
print(f"  1 ≤ |skew| < 2  (중간 비대칭) : {((sk >= 1) & (sk < 2)).sum()}개")
print(f"  |skew| ≥ 2    (강한 비대칭)  : {(sk >= 2).sum()}개")

# 비정규 컬럼 상위 목록 (왜도 기준)
print("\n[왜도 절대값 상위 10개 컬럼]")
top_skew = (
    results_df[['column', 'skewness', 'kurtosis', 'majority_normal']]
    .assign(abs_skew=results_df['skewness'].abs())
    .sort_values('abs_skew', ascending=False)
    .head(10)
)
print(top_skew[['column', 'skewness', 'kurtosis', 'majority_normal']].to_string(index=False))

print("\n" + "="*60)
print(f"💾 전체 결과 CSV: {OUTPUT_CSV}")
print("="*60)

▶ 데이터 로드 중...
  Shape: (28111, 256)
  정규성 검정 대상 컬럼 수: 250

▶ 정규성 검정 실행 중...
  진행: 50/250 (20.0%)
  진행: 100/250 (40.0%)
  진행: 150/250 (60.0%)
  진행: 200/250 (80.0%)
  진행: 250/250 (100.0%)

✅ 결과 저장 완료: normality_test_results.csv

📊 정규성 검정 요약

총 검정 컬럼 수   : 250
유의수준          : α = 0.05

[다수결 판정 - 3개 검정 중 2개 이상 정규]
  정규분포 O      : 0개  (0.0%)
  정규분포 X      : 250개  (100.0%)

[전체 합의 - 3개 검정 모두 정규]
  정규분포 O      : 0개  (0.0%)

[각 검정별 정규 비율]
  Shapiro-Wilk     : 0/250 (0.0%)
  Anderson-Darling : 0/250 (0.0%)
  D'Agostino-Pearson: 0/250 (0.0%)

[왜도(Skewness) 절대값 분포]
  |skew| < 0.5  (거의 대칭)   : 27개
  0.5 ≤ |skew| < 1 (약간 비대칭): 17개
  1 ≤ |skew| < 2  (중간 비대칭) : 41개
  |skew| ≥ 2    (강한 비대칭)  : 165개

[왜도 절대값 상위 10개 컬럼]
                column  skewness  kurtosis  majority_normal
                   종업원   21.8011  569.1265                0
       영업현금흐름증가율_ratio   13.7825  302.3695                0
유형자산회전율_ratio_industry    7.8827   76.9366                0
          유형자산회전율_diff    7.8577   75.3207      

## STEP1. Mann-Whitney U test

In [3]:
"""
피처셀렉션 v2 - Mann-Whitney U Test + FDR 보정 + 방향 구분 Effect Size
----------------------------------------------------------------------
① Mann-Whitney U test (비모수, 정규성 가정 없음)
② Benjamini-Hochberg FDR 보정 (다중검정 문제 해결)
③ CLES 방향 명시적 구분
     CLES > 0.55  → 부실기업이 높은 피처  (부채비율, 금융비용 등)
     CLES < 0.45  → 정상기업이 높은 피처  (수익성, 유동성 등)
     0.45 ~ 0.55  → 방향 불명확 → 제거
④ 최종 판단: FDR 유의 AND 방향 명확 → 유지
"""

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import warnings
import os

warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────────────────────────
DATA_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
OUTPUT_DIR = r'13번.피처셀렉션\M19_도매_소매업'
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'feature_selection_Utest_results.csv')

ALPHA         = 0.05   # FDR 보정 후 유의수준
CLES_HIGH_THR = 0.55   # 이 이상이면 부실기업이 높은 피처
CLES_LOW_THR  = 0.45   # 이 이하이면 정상기업이 높은 피처
               # 0.45 ~ 0.55 구간은 방향 불명확 → 제거

EXCLUDE_COLS = {
    '사업자등록번호', '회계년도', '회사명',
    '부실라벨_ICR3년', 'M코드', '빅4감사',
}

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
print("▶ 데이터 로드 중...")
df = pd.read_parquet(DATA_PATH)
print(f"  Shape: {df.shape}")

vc = df['부실라벨_ICR3년'].value_counts()
print(f"\n  [클래스 분포]")
print(f"  정상(0): {vc[0]:,}개 ({vc[0]/len(df)*100:.2f}%)")
print(f"  부실(1): {vc[1]:,}개 ({vc[1]/len(df)*100:.2f}%)")

normal_df   = df[df['부실라벨_ICR3년'] == 0]
distress_df = df[df['부실라벨_ICR3년'] == 1]

numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]
print(f"\n  검정 대상 컬럼: {len(numeric_cols)}개")


# ── 검정 함수 ─────────────────────────────────────────────────────────────────
def cohens_d(a, b):
    """Cohen's d — 양수면 부실>정상, 음수면 정상>부실"""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled_std = np.sqrt(
        ((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2)
    )
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std != 0 else 0.0


def run_test(col):
    a = distress_df[col].dropna().values.astype(float)  # 부실
    b = normal_df[col].dropna().values.astype(float)    # 정상

    result = {
        'column':          col,
        'n_distress':      len(a),
        'n_normal':        len(b),
        'mean_distress':   np.nan,
        'mean_normal':     np.nan,
        'median_distress': np.nan,
        'median_normal':   np.nan,
        'cohens_d':        np.nan,
        'cles':            np.nan,
        # 방향 판정
        'direction':       'insufficient_data',
        # 검정
        'mw_stat':         np.nan,
        'p_value_raw':     np.nan,
        'p_value_fdr':     np.nan,   # 나중에 채움
        'significant_fdr': np.nan,   # 나중에 채움
        # 최종
        'keep':            0,
        'remove_reason':   '',
    }

    if len(a) < 5 or len(b) < 5:
        result['remove_reason'] = '유효샘플 부족'
        return result

    result['mean_distress']   = round(float(np.mean(a)), 6)
    result['mean_normal']     = round(float(np.mean(b)), 6)
    result['median_distress'] = round(float(np.median(a)), 6)
    result['median_normal']   = round(float(np.median(b)), 6)
    result['cohens_d']        = round(float(cohens_d(a, b)), 4)

    # Mann-Whitney U
    try:
        u_stat, p_val = mannwhitneyu(a, b, alternative='two-sided')
        result['mw_stat']     = round(float(u_stat), 2)
        result['p_value_raw'] = float(p_val)
    except Exception as e:
        result['remove_reason'] = f'검정 오류: {e}'
        return result

    # CLES = P(부실 > 정상)
    cles_val = u_stat / (len(a) * len(b))
    result['cles'] = round(float(cles_val), 4)

    # 방향 판정
    if cles_val > CLES_HIGH_THR:
        result['direction'] = 'distress_high'    # 부실기업이 높음
    elif cles_val < CLES_LOW_THR:
        result['direction'] = 'normal_high'      # 정상기업이 높음
    else:
        result['direction'] = 'ambiguous'        # 방향 불명확

    return result


# ── 전체 실행 ─────────────────────────────────────────────────────────────────
print("\n▶ Mann-Whitney U 검정 실행 중...")
results = []
total = len(numeric_cols)

for i, col in enumerate(numeric_cols):
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"  진행: {i+1}/{total} ({(i+1)/total*100:.1f}%)")
    results.append(run_test(col))

results_df = pd.DataFrame(results)


# ── Benjamini-Hochberg FDR 보정 ───────────────────────────────────────────────
print("\n▶ FDR 보정(Benjamini-Hochberg) 적용 중...")

valid_mask = results_df['p_value_raw'].notna()
p_vals     = results_df.loc[valid_mask, 'p_value_raw'].values
n          = len(p_vals)

sorted_idx     = np.argsort(p_vals)
p_vals_sorted  = p_vals[sorted_idx]

# adjusted p-value (BH 공식)
adjusted = np.minimum(
    1,
    np.minimum.accumulate(
        (p_vals_sorted * n / np.arange(1, n + 1))[::-1]
    )[::-1]
)[np.argsort(sorted_idx)]

results_df.loc[valid_mask, 'p_value_fdr']   = adjusted
results_df.loc[valid_mask, 'significant_fdr'] = (adjusted <= ALPHA).astype(int)


# ── 최종 keep 판단 ────────────────────────────────────────────────────────────
# 유지 조건: FDR 유의 AND 방향 명확 (distress_high or normal_high)
for idx, row in results_df.iterrows():
    sig  = row['significant_fdr'] == 1
    dire = row['direction'] in ('distress_high', 'normal_high')

    if not sig and row['direction'] != 'insufficient_data':
        results_df.at[idx, 'remove_reason'] = 'FDR 비유의'
    elif sig and not dire:
        results_df.at[idx, 'remove_reason'] = '방향 불명확 (CLES 0.45~0.55)'
    elif sig and dire:
        results_df.at[idx, 'keep'] = 1


# ── 결과 저장 ─────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ 결과 저장 완료: {OUTPUT_CSV}")


# ── 요약 출력 ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("📊 피처셀렉션 v2 결과 요약 (방향 구분 적용)")
print("="*65)

n_total      = len(results_df)
n_sig_fdr    = int(results_df['significant_fdr'].sum())
n_keep       = int(results_df['keep'].sum())
n_remove     = n_total - n_keep

n_dist_high  = ((results_df['keep'] == 1) & (results_df['direction'] == 'distress_high')).sum()
n_norm_high  = ((results_df['keep'] == 1) & (results_df['direction'] == 'normal_high')).sum()

n_rm_fdr     = (results_df['remove_reason'] == 'FDR 비유의').sum()
n_rm_ambig   = (results_df['remove_reason'] == '방향 불명확 (CLES 0.45~0.55)').sum()
n_rm_data    = (results_df['remove_reason'] == '유효샘플 부족').sum()

print(f"\n총 검정 컬럼   : {n_total}개")
print(f"유의수준       : α = {ALPHA}")
print(f"CLES 임계값    : 부실↑ > {CLES_HIGH_THR}  |  정상↑ < {CLES_LOW_THR}  |  그 사이 → 불명확")

print(f"\n── 유지 ({n_keep}개) ──────────────────────────────────────")
print(f"  부실기업이 높은 피처 (CLES > {CLES_HIGH_THR})  : {n_dist_high}개")
print(f"  정상기업이 높은 피처 (CLES < {CLES_LOW_THR})  : {n_norm_high}개")

print(f"\n── 제거 ({n_remove}개) ──────────────────────────────────────")
print(f"  FDR 비유의                       : {n_rm_fdr}개")
print(f"  방향 불명확 (CLES 0.45~0.55)     : {n_rm_ambig}개")
print(f"  유효샘플 부족                    : {n_rm_data}개")

# 방향별 상위 피처
print(f"\n[부실기업이 높은 피처 상위 10개 — 부채, 비용 관련 예상]")
top_dist = (
    results_df[(results_df['keep'] == 1) & (results_df['direction'] == 'distress_high')]
    .sort_values('cles', ascending=False)
    .head(10)[['column', 'cles', 'cohens_d', 'median_distress', 'median_normal']]
)
print(top_dist.to_string(index=False))

print(f"\n[정상기업이 높은 피처 상위 10개 — 수익성, 유동성 관련 예상]")
top_norm = (
    results_df[(results_df['keep'] == 1) & (results_df['direction'] == 'normal_high')]
    .sort_values('cles', ascending=True)
    .head(10)[['column', 'cles', 'cohens_d', 'median_distress', 'median_normal']]
)
print(top_norm.to_string(index=False))

print(f"\n[방향 불명확으로 제거된 컬럼 목록]")
ambig_cols = results_df[results_df['remove_reason'] == '방향 불명확 (CLES 0.45~0.55)']['column'].tolist()
print(f"  {ambig_cols}")

print("\n" + "="*65)
print(f"💾 전체 결과: {OUTPUT_CSV}")
print("="*65)


# ── 유지 컬럼 목록 저장 ───────────────────────────────────────────────────────
U_test = results_df[results_df['keep'] == 1]['column'].tolist()

print(f"\n▶ U_test 변수에 유지 컬럼 {len(U_test)}개 저장 완료")
print(f"  예시: {U_test[:5]} ...")

▶ 데이터 로드 중...
  Shape: (11797, 256)

  [클래스 분포]
  정상(0): 11,343개 (96.15%)
  부실(1): 454개 (3.85%)

  검정 대상 컬럼: 250개

▶ Mann-Whitney U 검정 실행 중...
  진행: 50/250 (20.0%)
  진행: 100/250 (40.0%)
  진행: 150/250 (60.0%)
  진행: 200/250 (80.0%)
  진행: 250/250 (100.0%)

▶ FDR 보정(Benjamini-Hochberg) 적용 중...

✅ 결과 저장 완료: 13번.피처셀렉션\M19_도매_소매업\feature_selection_Utest_results.csv

📊 피처셀렉션 v2 결과 요약 (방향 구분 적용)

총 검정 컬럼   : 250개
유의수준       : α = 0.05
CLES 임계값    : 부실↑ > 0.55  |  정상↑ < 0.45  |  그 사이 → 불명확

── 유지 (202개) ──────────────────────────────────────
  부실기업이 높은 피처 (CLES > 0.55)  : 69개
  정상기업이 높은 피처 (CLES < 0.45)  : 133개

── 제거 (48개) ──────────────────────────────────────
  FDR 비유의                       : 25개
  방향 불명확 (CLES 0.45~0.55)     : 23개
  유효샘플 부족                    : 0개

[부실기업이 높은 피처 상위 10개 — 부채, 비용 관련 예상]
                 column   cles  cohens_d  median_distress  median_normal
          금융비용부담률_ratio 0.7745    0.7815         0.064960       0.012422
                금융비용부담률 0.7740    0.8333        

---
## STEP 2. LASSO (LassoCV)

**L1 정규화 회귀**로 불필요한 피처의 계수를 0으로 수렴시켜 자동 선택합니다.

- `LassoCV`: 교차검증으로 최적 alpha를 자동 결정
- 계수 ≠ 0 인 피처만 선택
- 표준화(`StandardScaler`) 적용 → 피처 간 스케일 차이 제거

> **alpha 의미**: 클수록 더 강한 정규화 → 더 많은 계수가 0이 됨 (피처 감소)

In [ ]:
"""
피처셀렉션 STEP 2 - LASSO (LassoCV)
------------------------------------
① U_test 결과에서 유지된 컬럼 사용
② LassoCV로 최적 alpha 자동 결정
③ 피처 수가 25 / 30 / 35개에 가장 가까운 alpha 각각 탐색 → 파일 저장
"""

import pandas as pd
import numpy as np
import warnings
import os
from sklearn.linear_model import LassoCV, lasso_path
warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────────────────────────
DATA_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
RESULT_PATH = r'13번.피처셀렉션\M19_도매_소매업\feature_selection_Utest_results.csv'
OUTPUT_DIR  = r'13번.피처셀렉션\M19_도매_소매업'

TARGET_COUNTS = [50,55,60,65]
CV_FOLDS      = 5
N_ALPHAS      = 100
MAX_ITER      = 10000
RANDOM_STATE  = 42

# ── 데이터 로드 ───────────────────────────────────────────────────────────────
print("▶ 데이터 로드 중...")
df = pd.read_parquet(DATA_PATH)

results_prev = pd.read_csv(RESULT_PATH)
U_test = results_prev[results_prev['keep'] == 1]['column'].tolist()

print(f"  전체 Shape  : {df.shape}")
print(f"  U_test 컬럼 : {len(U_test)}개")

X = df[U_test].copy()
y = df['부실라벨_ICR3년'].copy()

vc = y.value_counts()
print(f"\n  [클래스 분포]")
print(f"  정상(0): {vc[0]:,}개 ({vc[0]/len(y)*100:.2f}%)")
print(f"  부실(1): {vc[1]:,}개 ({vc[1]/len(y)*100:.2f}%)")

# ── NaN 처리 (중앙값 대체) ────────────────────────────────────────────────────
nan_total = X.isna().sum().sum()
print(f"\n  NaN 현황: {nan_total:,}개")
if nan_total > 0:
    nan_cols = X.isna().sum()
    nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
    print(f"  NaN 포함 컬럼 수: {len(nan_cols)}개")
    print(nan_cols.head(10).to_string())
    X = X.fillna(X.median())
    print(f"  → 중앙값 대체 완료 | 잔여 NaN: {X.isna().sum().sum()}")

# ── LassoCV로 최적 alpha 탐색 ─────────────────────────────────────────────────
print(f"\n▶ LassoCV 실행 중 (CV={CV_FOLDS}, alphas={N_ALPHAS})...")
lasso_cv = LassoCV(
    cv           = CV_FOLDS,
    n_alphas     = N_ALPHAS,
    max_iter     = MAX_ITER,
    random_state = RANDOM_STATE,
    n_jobs       = -1,
)
lasso_cv.fit(X, y)
best_alpha = lasso_cv.alpha_
print(f"  LassoCV 최적 alpha: {best_alpha:.6f}")

# ── alpha 그리드 탐색으로 피처 수 조절 ───────────────────────────────────────
alpha_grid = np.logspace(
    np.log10(best_alpha * 0.001),
    np.log10(best_alpha * 10),
    2000
)

alphas_out, coefs_out, _ = lasso_path(X, y, alphas=alpha_grid[::-1], max_iter=MAX_ITER)
# coefs_out shape: (n_features, n_alphas)

alpha_to_count = {a: (coefs_out[:, i] != 0).sum() for i, a in enumerate(alphas_out)}
alpha_to_coef  = {a: coefs_out[:, i] for i, a in enumerate(alphas_out)}

# ── 타겟별 최적 alpha 선택 & 저장 ────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "=" * 65)
print("📊 타겟 피처 수별 결과")
print("=" * 65)

for target in TARGET_COUNTS:
    best_a       = min(alpha_to_count, key=lambda a: abs(alpha_to_count[a] - target))
    actual_count = alpha_to_count[best_a]
    coef         = alpha_to_coef[best_a]

    coef_df = pd.DataFrame({
        'column'   : U_test,
        'coef'     : coef,
        'abs_coef' : np.abs(coef),
        'selected' : (coef != 0).astype(int),
    }).sort_values('abs_coef', ascending=False).reset_index(drop=True)

    selected_cols = coef_df[coef_df['selected'] == 1]['column'].tolist()

    csv_path = os.path.join(OUTPUT_DIR, f'lasso_results_top{target}.csv')
    col_path = os.path.join(OUTPUT_DIR, f'lasso_features_top{target}.csv')

    coef_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    pd.DataFrame({'column': selected_cols}).to_csv(col_path, index=False, encoding='utf-8-sig')

    print(f"\n  [타겟 {target}개]")
    print(f"    최적 alpha    : {best_a:.6f}")
    print(f"    실제 피처 수  : {actual_count}개")
    print(f"    계수 결과     : {csv_path}")
    print(f"    피처 목록     : {col_path}")
    print(f"    상위 10개     : {selected_cols[:10]}")

print("\n" + "=" * 65)
print("✅ 전체 저장 완료")
print("=" * 65)

▶ 데이터 로드 중...
  전체 Shape  : (28111, 256)
  U_test 컬럼 : 202개

  [클래스 분포]
  정상(0): 27,056개 (96.25%)
  부실(1): 1,055개 (3.75%)

  NaN 현황: 72개
  NaN 포함 컬럼 수: 3개
업력          24
업력_diff     24
업력_ratio    24
  → 중앙값 대체 완료 | 잔여 NaN: 0

▶ LassoCV 실행 중 (CV=5, alphas=100)...
  LassoCV 최적 alpha: 0.000030

📊 타겟 피처 수별 결과

  [타겟 55개]
    최적 alpha    : 0.000085
    실제 피처 수  : 55개
    계수 결과     : 13번.피처셀렉션\M19_도매_소매업\lasso_results_top55.csv
    피처 목록     : 13번.피처셀렉션\M19_도매_소매업\lasso_features_top55.csv
    상위 10개     : ['금융비용부담률', '총자본영업이익률_diff', '영업CF_총부채_diff', 'ROA변화', 'ROA_ratio', '총자본영업이익률', '현금ROA', 'ROE_diff', '매출총이익률_diff', '매입채무지급기간_diff']

  [타겟 60개]
    최적 alpha    : 0.000055
    실제 피처 수  : 60개
    계수 결과     : 13번.피처셀렉션\M19_도매_소매업\lasso_results_top60.csv
    피처 목록     : 13번.피처셀렉션\M19_도매_소매업\lasso_features_top60.csv
    상위 10개     : ['금융비용부담률', '총자본영업이익률_diff', '매출총이익률_diff', '영업CF_총부채_diff', 'ROA_ratio', 'ROA변화', '현금ROA', '총자본영업이익률', '매출원가율', 'ROE_diff']

  [타겟 65개]
    최적 alpha    : 0.000045

## 유사한 칼럼은 지우고, 상단만 남길것

In [6]:
"""
dedupe_columns_nb.py
────────────────────
ipynb 전용 버전 — argparse 없이 셀에서 바로 실행
"""

import re
import glob as glob_module
import pandas as pd
from pathlib import Path

# ── suffix 패턴 ───────────────────────────────────────────────────────────────
SUFFIX_PATTERNS = [
    r"_ratio_industry$",
    r"_diff_industry$",
    r"_diff$",
    r"_ratio$",
    r"_industry$",
]

def get_base(col: str) -> str:
    for pat in SUFFIX_PATTERNS:
        stripped = re.sub(pat, "", col)
        if stripped != col:
            return stripped
    return col

def dedupe_columns(columns: list) -> tuple:
    seen = {}
    kept, removed = [], []
    for col in columns:
        base = get_base(col)
        if base not in seen:
            seen[base] = col
            kept.append(col)
        else:
            removed.append((col, seen[base]))
    return kept, removed

def load_columns(path: Path) -> list:
    peek = pd.read_csv(path, nrows=1, header=None)
    first_val = str(peek.iloc[0, 0]).strip()
    if re.match(r"^[\d\.]+$", first_val):
        df = pd.read_csv(path, header=None)
    else:
        df = pd.read_csv(path, header=0)
    cols = df.iloc[:, 0].dropna().astype(str).str.strip().tolist()
    return [c for c in cols if c]

def process_file(path, verbose=True) -> Path:
    path = Path(path)
    columns = load_columns(path)
    kept, removed = dedupe_columns(columns)

    n_kept    = len(kept)
    n_removed = len(removed)

    out_name = f"{path.stem}--{n_kept}.csv"
    out_path = path.parent / out_name

    pd.DataFrame(kept, columns=["feature"]).to_csv(
        out_path, index=False, encoding="utf-8-sig"
    )

    if verbose:
        print(f"\n{'─'*55}")
        print(f"  파일  : {path.name}")
        print(f"  원본  : {len(columns)}개  →  유지: {n_kept}개  제거: {n_removed}개")
        print(f"  저장  : {out_path}")
        if removed:
            print(f"\n  [제거된 칼럼]")
            for rm, rep in removed:
                print(f"    {rm:<40} → {rep}")
    return out_path

def process_glob(pattern: str, verbose=True):
    """글로브 패턴으로 여러 파일 한번에 처리"""
    paths = sorted(Path(p) for p in glob_module.glob(pattern))
    if not paths:
        print(f"[경고] 해당 패턴에 맞는 파일 없음: {pattern}")
        return []
    print(f"\n총 {len(paths)}개 파일 처리 시작")
    results = []
    for p in paths:
        out = process_file(p, verbose=verbose)
        results.append(out)
    print(f"\n{'='*55}")
    print(f"  완료: {len(results)}개 파일 저장")
    for r in results:
        print(f"    → {r}")
    return results

In [7]:
files = [
    r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55.csv",
    r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top60.csv",
    r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top65.csv"
]
for f in files:
    process_file(f)


───────────────────────────────────────────────────────
  파일  : lasso_features_top55.csv
  원본  : 55개  →  유지: 45개  제거: 10개
  저장  : 13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv

  [제거된 칼럼]
    총자본영업이익률                                 → 총자본영업이익률_diff
    매입채무지급기간_ratio                           → 매입채무지급기간_diff
    매입채무지급기간                                 → 매입채무지급기간_diff
    현금ROA_ratio                              → 현금ROA
    유형자산비율_ratio                             → 유형자산비율
    ROIC_ratio                               → ROIC_diff
    ROA변화_ratio                              → ROA변화
    현금ROE                                    → 현금ROE_ratio
    차입금의존도_ratio_industry                    → 차입금의존도_diff_industry
    매출액순이익률_ratio_industry                   → 매출액순이익률_diff_industry

───────────────────────────────────────────────────────
  파일  : lasso_features_top60.csv
  원본  : 60개  →  유지: 49개  제거: 11개
  저장  : 13번.피처셀렉션\M19_도매_소매업\lasso_features_top60--49.csv

  [제거된 칼럼]
    총자본영업이익률      